In [1]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# from recbole.config import Config
# from recbole.data import create_dataset, data_preparation
from UniSRec.unisrec import UniSRec
# IMPORTANT: register UniSRec custom dataset
from UniSRec.data.dataset import UniSRecDataset
import sys
import os

sys.path.insert(0, os.path.abspath("UniSRec"))



In [2]:
from pathlib import Path

PROJECT_ROOT = Path.home() / "RecSys"
DATASET_NAME = "All_Beauty"

UNISREC_DIR = PROJECT_ROOT / "UniSRec"
PROPS_DIR = UNISREC_DIR / "props"
DATASET_DIR = UNISREC_DIR / "dataset" / DATASET_NAME
SAVED_DIR = UNISREC_DIR / "saved"
META_JSONL = PROJECT_ROOT / "meta_All_Beauty.jsonl"

print("UNISREC_DIR:", UNISREC_DIR, UNISREC_DIR.exists())
print("PROPS_DIR:", PROPS_DIR, PROPS_DIR.exists())
print("DATASET_DIR (before):", DATASET_DIR, DATASET_DIR.exists())

DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("DATASET_DIR (after):", DATASET_DIR, DATASET_DIR.exists())

UNISREC_DIR: /home/yonataba/RecSys/UniSRec True
PROPS_DIR: /home/yonataba/RecSys/UniSRec/props True
DATASET_DIR (before): /home/yonataba/RecSys/UniSRec/dataset/All_Beauty True
DATASET_DIR (after): /home/yonataba/RecSys/UniSRec/dataset/All_Beauty True


In [23]:
# --- UniSRec model config ---
unisrec_yaml = """
model: UniSRec

# ===== REQUIRED ARCH PARAMS (must not be None) =====
hidden_act: gelu
initializer_range: 0.02
hidden_size: 300
embedding_size: 300
plm_size: 768
n_layers: 2
n_heads: 2
inner_size: 256
hidden_dropout_prob: 0
attn_dropout_prob: 0
layer_norm_eps: 1e-12

# ===== UniSRec-specific =====
n_exps: 8
adaptor_layers: [768, 300]
adaptor_dropout_prob: 0.05
temperature: 0.07
item_drop_ratio: 0.1
lambda: 0.02
loss_type: CE
train_stage: transductive_ft
plm_suffix: None



# ===== data =====
load_col:
  inter: [user_id, item_id, timestamp]

USER_ID_FIELD: user_id
ITEM_ID_FIELD: item_id
TIME_FIELD: timestamp

train_neg_sample_args: ~

# ===== training =====
train_batch_size: 512
epochs: 100
stopping_step: 15
plm_suffix: None
plm_suffix_aug: None

device: cuda

""".strip()

# --- Finetune config ---
finetune_yaml = """
model: UniSRec

load_col:
  inter: [user_id, item_id, timestamp]

USER_ID_FIELD: user_id
ITEM_ID_FIELD: item_id
TIME_FIELD: timestamp

alias_of_item_id: ~
enable_scaler: False

loss_type: CE
train_neg_sample_args: ~

learning_rate: 2e-5
train_batch_size: 512
epochs: 100
stopping_step: 15
eval_step: 1
pretrained_model_path: saved/UniSRec-best-17-12.pth


device: cuda
""".strip()

# --- Dataset config ---
dataset_yaml = """
data_path: UniSRec/dataset
benchmark_filename: [train, valid]
load_col:
  inter: [user_id, item_id_list, item_length, item_id]

# Tell RecBole how to interpret your columns
alias_of_item_id: [item_id]
alias_of_item_seq: [item_id_list]
alias_of_item_seq_len: [item_length]

USER_ID_FIELD: user_id
ITEM_ID_FIELD: item_id
TIME_FIELD: timestamp
MAX_ITEM_LIST_LENGTH: 50
min_user_inter: 1
user_inter_num_interval: "[1,inf)"
min_item_list_len: 1



train_neg_sample_args: ~

eval_args:
  order: TO
  mode:
    valid: full


eval_step: 1
metrics: [Recall]
topk: [10, 50]
valid_metric: Recall@50
eval_batch_size: 2048
""".strip()

(PROPS_DIR / "UniSRec.yaml").write_text(unisrec_yaml)
(PROPS_DIR / "finetune.yaml").write_text(finetune_yaml)
(PROPS_DIR / "All_Beauty.yaml").write_text(dataset_yaml)

for p in ["UniSRec.yaml", "finetune.yaml", "All_Beauty.yaml"]:
    path = PROPS_DIR / p
    print(p, "exists:", path.exists(), "size:", path.stat().st_size if path.exists() else None)

UniSRec.yaml exists: True size: 768
finetune.yaml exists: True size: 368
All_Beauty.yaml exists: True size: 609


In [4]:
import pandas as pd
from pathlib import Path

DATASET_DIR = Path("UniSRec/dataset/All_Beauty")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = "All_Beauty.train.csv"
VALID_CSV = "All_Beauty.valid.csv"

# Truncate to 2000 lines for debugging
train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)


def build_list_inter(df):
    rows = []
    MAX_LEN = 50

    for _, r in df.iterrows():
        if isinstance(r["history"], str) and r["history"].strip() != "":
            hist_items = r["history"].split()
            hist_items = hist_items[-MAX_LEN:]   
        else:
            hist_items = []

        rows.append({
            "user_id": r["user_id"],
            "item_id_list": " ".join(hist_items),
            "item_id": r["parent_asin"]
        })

    return pd.DataFrame(rows)



train_inter = build_list_inter(train_df)
valid_inter = build_list_inter(valid_df)

print("Train rows:", train_inter.shape)
print("Valid rows:", valid_inter.shape)

Train rows: (583190, 3)
Valid rows: (71784, 3)


In [5]:
def write_inter(df, path):
    with open(path, "w") as f:
        f.write("user_id:token\titem_id_list:token_seq\titem_id:token\n")
        df.to_csv(f, sep="\t", index=False, header=False)


write_inter(train_inter, DATASET_DIR / "All_Beauty.train.inter")
write_inter(valid_inter, DATASET_DIR / "All_Beauty.valid.inter")

print("Wrote list-based benchmark files")


Wrote list-based benchmark files


In [6]:
item_text = {}

with open(META_JSONL) as f:
    for line in f:
        j = json.loads(line)

        asin = j.get("parent_asin")
        if asin is None:
            continue

        def to_str(x):
            if isinstance(x, list):
                return " ".join(map(str, x))
            return str(x) if x is not None else ""

        text = " ".join([
            to_str(j.get("title")),
            to_str(j.get("store")),
        ]).strip()

        if text:
            item_text[asin] = text

len(item_text)


112588

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "sentence-transformers/all-distilroberta-v1"  # 768-dim
model = SentenceTransformer(MODEL_NAME, device="cuda")

texts = list(item_text.values())

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", embeddings.shape)  # (num_items, 768)
embeddings = embeddings.astype(np.float32)
embeddings.tofile(DATASET_DIR / "All_Beauty.None")


Batches:   0%|          | 0/1760 [00:00<?, ?it/s]

Embeddings shape: (112588, 768)


In [59]:
import fasttext
import numpy as np

ft = fasttext.load_model("cc.en.300.bin")

texts = list(item_text.values())

def embed(text):
    words = text.lower().split()
    if not words:
        return np.zeros(300, dtype=np.float32)
    vecs = [ft.get_word_vector(w) for w in words]
    v = np.mean(vecs, axis=0)
    return v / (np.linalg.norm(v) + 1e-8)

embeddings = np.stack([embed(t) for t in texts]).astype(np.float32)

print(embeddings.shape)  # (num_items, 300)

embeddings.tofile(DATASET_DIR / "All_Beauty.None")


(112588, 300)


In [24]:
import numpy as np

from recbole.config import Config
from UniSRec.unisrec import UniSRec
from UniSRec.data.dataset import UniSRecDataset

config = Config(
    model=UniSRec,
    dataset=DATASET_NAME,
    config_file_list=[
        str(PROPS_DIR / "UniSRec.yaml"),
        str(PROPS_DIR / "finetune.yaml"),
        str(PROPS_DIR / "All_Beauty.yaml"),
    ],
    config_dict={
        "dataset_class": UniSRecDataset,
        "dataset_path": str(UNISREC_DIR / "dataset" / DATASET_NAME),
    }
)

# 🔴 critical overrides
# 🔧 Force RecBole to use directory-based dataset layout
# config["dataset_path"] = str(UNISREC_DIR / "dataset" / DATASET_NAME)

print("config['dataset_path']:", config["dataset_path"])

base = Path(config["dataset_path"])

for split in ["train", "valid"]:
    p = base / f"{DATASET_NAME}.{split}.inter"
    print(p, "exists:", p.exists())


config["eval_args"] = {
    "order": "TO",
    "mode": {
        "valid": "full"
    }
}

print("eval_args AFTER override =", config["eval_args"])

config['dataset_path']: /home/yonataba/RecSys/UniSRec/dataset/All_Beauty
/home/yonataba/RecSys/UniSRec/dataset/All_Beauty/All_Beauty.train.inter exists: True
/home/yonataba/RecSys/UniSRec/dataset/All_Beauty/All_Beauty.valid.inter 

exists: True
eval_args AFTER override = {'order': 'TO', 'mode': {'valid': 'full'}}


In [25]:
from UniSRec.data.dataset import UniSRecDataset

dataset = UniSRecDataset(config)

print(type(dataset))

train_dataset, valid_dataset = dataset.build()

print(type(train_dataset))
print(hasattr(train_dataset, "plm_embedding"))
print(train_dataset.plm_embedding.weight.shape)


/home/yonataba/.conda/envs/k8/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value="", inplace=True)
/home/yonataba/.conda/envs/k8/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are

<class 'UniSRec.data.dataset.UniSRecDataset'>
<class 'UniSRec.data.dataset.UniSRecDataset'>
True
torch.Size([107642, 768])


In [27]:
from recbole.data.dataloader import TrainDataLoader, FullSortEvalDataLoader
from recbole.sampler import Sampler
# Filter TRAIN ONLY
# Filter TRAIN
train_dataset.inter_feat = train_dataset.inter_feat[
    train_dataset.inter_feat["item_length"] > 0
]

# Filter VALID
valid_dataset.inter_feat = valid_dataset.inter_feat[
    valid_dataset.inter_feat["item_length"] > 0
]

train_data = TrainDataLoader(
    config, train_dataset, Sampler(config, train_dataset)
)

valid_data = FullSortEvalDataLoader(
    config, valid_dataset, Sampler(config, valid_dataset)
)



model = UniSRec(config, train_dataset).to(config["device"])
for param in model.position_embedding.parameters():
    param.requires_grad = False
for param in model.trm_encoder.parameters():
    param.requires_grad = False
for p in model.plm_embedding.parameters():
    p.requires_grad = False

In [29]:
# ❄️ Freeze item embedding (CRITICAL)
for p in model.item_embedding.parameters():
    p.requires_grad = False

# ❄️ Freeze PLM embedding
for p in model.plm_embedding.parameters():
    p.requires_grad = False

# ❄️ Freeze transformer encoder
for p in model.trm_encoder.parameters():
    p.requires_grad = False

# ❄️ Freeze position embedding
for p in model.position_embedding.parameters():
    p.requires_grad = False

# (Optional but recommended) Freeze final LayerNorm
for p in model.LayerNorm.parameters():
    p.requires_grad = False
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("\n".join(trainable))
print("Num trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


moe_adaptor.w_gate
moe_adaptor.w_noise
moe_adaptor.experts.0.bias
moe_adaptor.experts.0.lin.weight
moe_adaptor.experts.1.bias
moe_adaptor.experts.1.lin.weight
moe_adaptor.experts.2.bias
moe_adaptor.experts.2.lin.weight
moe_adaptor.experts.3.bias
moe_adaptor.experts.3.lin.weight
moe_adaptor.experts.4.bias
moe_adaptor.experts.4.lin.weight
moe_adaptor.experts.5.bias
moe_adaptor.experts.5.lin.weight
moe_adaptor.experts.6.bias
moe_adaptor.experts.6.lin.weight
moe_adaptor.experts.7.bias
moe_adaptor.experts.7.lin.weight
Num trainable params: 1861632


In [11]:
inter = train_data.dataset.inter_feat.interaction

print(inter['item_id_list'].shape)   # (?, 50)
print(inter['item_length'].max())    # 50
print(model.position_embedding.weight.shape[0])  # 50


torch.Size([48389, 50])
tensor(50)
50


In [10]:
print("plm_embedding:", model.plm_embedding.weight.shape)      # expect (num_items, 768)
print("w_gate:", model.moe_adaptor.w_gate.shape)              # expect (768, 8)
print("expert0:", model.moe_adaptor.experts[0].lin.weight.shape)  # expect (300, 768)


plm_embedding: torch.Size([107642, 768])
w_gate: torch.Size([768, 8])
expert0: torch.Size([300, 768])


In [14]:
config


General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = UniSRec/dataset/All_Beauty
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 100
train_batch_size = 512
learner = adam
learning_rate = 0.0001
train_neg_sample_args = {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 15
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'order': 'TO', 'mode': {'valid': 'full'}}
repeatable = True
metrics = ['Recall']
topk = [10, 50]
valid_metric = Recall@50
valid_metric_bigger = True
eval_batch_size = 2048
metric_decimal_place = 4

Dataset Hyper Parameters:
field_separator = 	
seq_separator =  
USER_ID_FIELD = user_id
ITEM_ID_FIELD = item_id
RATING_FIELD = ratin

In [13]:
# --- Print Model Parameters Before Loading ---
print("Model parameters BEFORE loading pre-trained weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")

Model parameters BEFORE loading pre-trained weights:
item_embedding.weight: torch.Size([107642, 300])
position_embedding.weight: torch.Size([50, 300])
trm_encoder.layer.0.multi_head_attention.query.weight: torch.Size([300, 300])
trm_encoder.layer.0.multi_head_attention.query.bias: torch.Size([300])
trm_encoder.layer.0.multi_head_attention.key.weight: torch.Size([300, 300])
trm_encoder.layer.0.multi_head_attention.key.bias: torch.Size([300])
trm_encoder.layer.0.multi_head_attention.value.weight: torch.Size([300, 300])
trm_encoder.layer.0.multi_head_attention.value.bias: torch.Size([300])
trm_encoder.layer.0.multi_head_attention.dense.weight: torch.Size([300, 300])
trm_encoder.layer.0.multi_head_attention.dense.bias: torch.Size([300])
trm_encoder.layer.0.multi_head_attention.LayerNorm.weight: torch.Size([300])
trm_encoder.layer.0.multi_head_attention.LayerNorm.bias: torch.Size([300])
trm_encoder.layer.0.feed_forward.dense_1.weight: torch.Size([256, 300])
trm_encoder.layer.0.feed_forward.

In [11]:
print("plm_embedding size:", model.plm_embedding.weight.shape[0])
print("item_embedding size:", model.item_embedding.weight.shape[0])
print("max item_id in dataset:", dataset.item_num - 1)
print("max item_id in train:", train_data.dataset.inter_feat['item_id'].max())



plm_embedding size: 107642
item_embedding size: 107642
max item_id in dataset: 107641
max item_id in train: tensor(95431)


In [ ]:
# --- Load Pre-trained Weights ---
pretrained_path = "UniSRec-FHCKM-300.pth"

checkpoint = torch.load(
    pretrained_path,
    map_location=config["device"],
    weights_only=False

)

sd = checkpoint["state_dict"]

missing, unexpected = model.load_state_dict(sd, strict=False)

print("Missing:", missing)
print("Unexpected:", unexpected)




# Optional: Freeze encoder parameters (common in fine-tuning)
# Uncomment the lines below if you want to freeze the transformer encoder
for param in model.position_embedding.parameters():
    param.requires_grad = False
for param in model.trm_encoder.parameters():
    param.requires_grad = False
    for p in model.plm_embedding.parameters():
    p.requires_grad = False
# print("Encoder parameters frozen.")
    


Missing: ['item_embedding.weight', 'plm_embedding.weight']
Unexpected: []


In [13]:
print("Transformer loaded:",
      model.trm_encoder.layer[0].feed_forward.dense_1.weight.norm().item())

print("MoE gate shape:", model.moe_adaptor.w_gate.shape)
print("MoE expert shape:", model.moe_adaptor.experts[0].lin.weight.shape)


Transformer loaded: 54.9044303894043
MoE gate shape: torch.Size([768, 8])
MoE expert shape: torch.Size([300, 768])


In [ ]:
from recbole.trainer import Trainer
from tqdm.notebook import tqdm
from recbole.utils import init_logger

init_logger(config)



trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(
    train_data, valid_data, saved=True, show_progress=False
)


/home/yonataba/.conda/envs/k8/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
17 Dec 20:08    INFO  epoch 0 training [time: 16.73s, train loss: 1135.6086]
17 Dec 20:08    INFO  epoch 0 evaluating [time: 0.45s, valid_score: 0.000000]
17 Dec 20:08    INFO  valid result: 
recall@10 : 0.0    recall@50 : 0.0
17 Dec 20:08    INFO  Saving current: saved/UniSRec-Dec-17-2025_20-08-39.pth
17 Dec 20:09    INFO  epoch 1 training [time: 17.70s, train loss: 1127.4207]
17 Dec 20:09    INFO  epoch 1 evaluating [time: 0.50s, valid_score: 0.000000]
17 Dec 20:09    INFO  valid result: 
recall@10 : 0.0    recall@50 : 0.0
17 Dec 20:09    INFO  Saving current: saved/UniSRec-Dec-17-2025_20-08-39.pth
17 Dec 20:09    INFO  epoch 2 training [time: 17.10s, train loss: 1121.4681]
17 Dec 20:09    INFO  epoch 2 evaluating [t

In [9]:
print("valid users:", len(valid_dataset.inter_feat))


valid users: 8776


In [10]:
# Final validation results (no test set per professor's requirement)
print("Final Validation Results:")
print(f"Best Recall@10: {best_valid_result['recall@50']}")


Final Validation Results:
Best Recall@10: 0.0174


In [12]:
import pandas as pd

test_df = pd.read_csv("test.csv")

rows = []
for _, row in test_df.iterrows():
    uid = row["id"]
    history = str(row.get("history", "")).strip()
    items = history.split()

    for t, it in enumerate(items):
        rows.append({
            "user_id": uid,
            "item_id": it,
            "timestamp": t
        })

test_inter = pd.DataFrame(rows)
test_inter.to_csv("UniSRec/dataset/All_Beauty/All_Beauty.test.inter", index=False)

print("Wrote test.inter with", len(test_inter), "rows")


Wrote test.inter with 16996 rows


In [17]:
for name, param in model.named_parameters():
    print(name)

item_embedding.weight
position_embedding.weight
trm_encoder.layer.0.multi_head_attention.query.weight
trm_encoder.layer.0.multi_head_attention.query.bias
trm_encoder.layer.0.multi_head_attention.key.weight
trm_encoder.layer.0.multi_head_attention.key.bias
trm_encoder.layer.0.multi_head_attention.value.weight
trm_encoder.layer.0.multi_head_attention.value.bias
trm_encoder.layer.0.multi_head_attention.dense.weight
trm_encoder.layer.0.multi_head_attention.dense.bias
trm_encoder.layer.0.multi_head_attention.LayerNorm.weight
trm_encoder.layer.0.multi_head_attention.LayerNorm.bias
trm_encoder.layer.0.feed_forward.dense_1.weight
trm_encoder.layer.0.feed_forward.dense_1.bias
trm_encoder.layer.0.feed_forward.dense_2.weight
trm_encoder.layer.0.feed_forward.dense_2.bias
trm_encoder.layer.0.feed_forward.LayerNorm.weight
trm_encoder.layer.0.feed_forward.LayerNorm.bias
trm_encoder.layer.1.multi_head_attention.query.weight
trm_encoder.layer.1.multi_head_attention.query.bias
trm_encoder.layer.1.multi_

In [13]:
test_df.head(50)

,Unnamed: 0,id,history
0,0,0,B0020MKBNW B082FLP15V B00946HGLW
1,1,1,B00N6WHTRG B00NNKWDI6 B00MDKICPK B010B0S67C B0...
2,2,2,B00N6WHTRG B00NNKWDI6 B00MDKICPK B010B0S67C B0...
3,3,3,B00N6WHTRG B00NNKWDI6 B00MDKICPK B010B0S67C B0...
4,4,4,B07FM69672 B07J3GH1W1 B07F8PCLDV B071JMGPTH B0...
5,5,5,B01BEYRHBA B071NBFSLY B073WQHFXB B07DPKZPRH B0...
6,6,6,B0974GWS7Z
7,7,7,B089RLLT5C B087Z9X39L B07ZS3DKL5 B08CL46XNM B0...
8,8,8,B089RLLT5C B087Z9X39L B07ZS3DKL5 B08CL46XNM B0...
9,9,9,B089RLLT5C B087Z9X39L B07ZS3DKL5 B08CL46XNM B0...


In [15]:
import torch
import pandas as pd
from tqdm import tqdm
from recbole.data import create_dataset, data_preparation
from recbole.data.interaction import Interaction

# ------------------------------
# 1. Load dataset (RecBole)
# ------------------------------

uid_field = dataset.uid_field
iid_field = dataset.iid_field

token2id_item = dataset.field2token_id[iid_field]
id2token_item = lambda x: dataset.id2token(iid_field, x)

df_test = pd.read_csv("test.csv")

# SASRec sequence fields
item_seq_field = model.ITEM_SEQ       # "item_id_list"
item_seq_len_field = model.ITEM_SEQ_LEN  # "item_length"
max_len = config['MAX_ITEM_LIST_LENGTH']

device = model.device
top_k = 10

model.eval()

print(f"🔥 Running batched inference on device: {device}")
print(f"🧠 Max sequence length: {max_len}")

# ------------------------------------------
# 2. Preprocess all histories
# ------------------------------------------
seqs = []
seq_lens = []
seen_sets = []
session_ids = df_test["id"].tolist()

for _, row in df_test.iterrows():
    history_asins = row["history"].split()

    internal_hist = [token2id_item[a] for a in history_asins if a in token2id_item]
    if len(internal_hist) == 0:
        internal_hist = [0]

    seen_sets.append(set(internal_hist))

    seq = internal_hist[-max_len:]
    seq_len = len(seq)
    padding = [0] * (max_len - seq_len)
    seq_padded = seq + padding

    seqs.append(seq_padded)
    seq_lens.append(seq_len)

# convert to GPU tensors
seqs = torch.tensor(seqs, dtype=torch.long).to(device)
seq_lens = torch.tensor(seq_lens, dtype=torch.long).to(device)
uids = torch.zeros(len(df_test), dtype=torch.long).to(device)  # dummy

# ------------------------------------------
# 3. Predict in batches
# ------------------------------------------
batch_size = 512
all_scores = []

print("🚀 Predicting in batches...")

with torch.no_grad():
    for i in tqdm(range(0, len(df_test), batch_size)):
        b_seq = seqs[i:i+batch_size]
        b_len = seq_lens[i:i+batch_size]
        b_uid = uids[i:i+batch_size]

        interaction = Interaction({
            uid_field: b_uid,
            item_seq_field: b_seq,
            item_seq_len_field: b_len
        })

        scores = model.full_sort_predict(interaction)
        all_scores.append(scores)

scores = torch.cat(all_scores, dim=0)

# ------------------------------------------
# 4. Mask seen items
# ------------------------------------------
for idx, seen in enumerate(seen_sets):
    if seen:
        scores[idx, list(seen)] = -1e9

# ------------------------------------------
# 5. Top-K and convert to ASINs
# ------------------------------------------
topk_indices = torch.topk(scores, top_k, dim=1).indices.cpu().tolist()

# ------------------------------------------
# 6. Convert to required Kaggle format
# ------------------------------------------
rows = []

for session_id, internal_items in zip(session_ids, topk_indices):
    asin_items = [id2token_item(i) for i in internal_items]
    row = {"id": session_id}
    for j in range(10):
        row[f"rec{j+1}"] = asin_items[j]
    rows.append(row)

submission = pd.DataFrame(rows)
submission.to_csv("submission.csv", index=False)

print("✔ submission.csv written in correct format!")
submission.head()


🔥 Running batched inference on device: cuda
🧠 Max sequence length: 170
🚀 Predicting in batches...


100%|██████████| 10/10 [00:00<00:00, 32.37it/s]


✔ submission.csv written in correct format!


,id,rec1,rec2,rec3,rec4,rec5,rec6,rec7,rec8,rec9,rec10
0,0,B00N6WHTRG,B01LX46XC2,B00X6QE9MK,B07BRS6VFP,B01487LVUI,B0BNWSR278,B07SRW6NVK,B00KCTER3U,B07J3GH1W1,B00DT4757A
1,1,B08QFLW9B2,B09GNXK3N1,B08MTW68VR,B08G4J3KDL,B094TYN9WW,B08D7TLV21,B08R8SGWVJ,B0863FZFPV,B08XZT4FLY,B08YYY2Y2X
2,2,B08QFLW9B2,B09GNXK3N1,B08MTW68VR,B08G4J3KDL,B094TYN9WW,B08D7TLV21,B08R8SGWVJ,B0863FZFPV,B08XZT4FLY,B08YYY2Y2X
3,3,B08QFLW9B2,B09GNXK3N1,B08MTW68VR,B08G4J3KDL,B094TYN9WW,B08D7TLV21,B08R8SGWVJ,B0863FZFPV,B08XZT4FLY,B08YYY2Y2X
4,4,B08QHP717Z,B0B2L218H2,B0BTJ6SYKB,B09GNXK3N1,B08LPGZMQK,B08P4YQ3K5,B08RS762Z7,B086TS3BKQ,B01MA3LXIL,B08DXCVRNY


In [130]:
import os

print("HOME:", os.path.expanduser("~"))
print("kaggle.json exists:",
      os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")))
print("CWD:", os.getcwd())


HOME: /home/yonataba
kaggle.json exists: True
CWD: /home/yonataba/RecSys


In [131]:
import json, os

path = os.path.expanduser("~/.kaggle/kaggle.json")
with open(path) as f:
    creds = json.load(f)

print(creds.keys())
print("username:", creds.get("username"))
print("key starts with:", creds.get("key", "")[:5])


dict_keys(['username', 'key'])
username: yonatanbaruchbaruch
key starts with: KGAT_
